
# V5 — Combined Pre-pair Necessary-condition Gate

**Purpose:** combine the already frozen Original DEV coverage and E1 frozen coverage **without inspecting any 1–5 year gap**, and decide whether local pairing is even mathematically feasible under the frozen minimums.

This notebook must **not**:
- inspect pair gaps,
- generate pairs,
- rebalance chronology,
- generate astrology,
- score Control,
- research CONFIRM,
- lower the frozen thresholds.

If the combined upper bound still fails, the only allowed action is a **whole new outcome-blind DEV expansion wave E2** under the same `C40 / P50 / S70` design.


In [1]:

from pathlib import Path
import json, hashlib
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V5_COMBINED_PREPAIR_FEASIBILITY_GATE_20260817"

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def find_repo_root():
    starts = [Path.cwd(), Path.cwd().resolve()]
    for start in starts:
        for p in [start] + list(start.parents):
            if (p / "research" / "ml").exists():
                return p
    raise RuntimeError("Could not locate repo root containing research/ml")

ROOT = find_repo_root()
OUT = ROOT / "research" / "ml" / "artifacts" / "v5_combined_prepair_feasibility"
OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_PATH = (
    ROOT / "research" / "ml_corpus" / "v5_ground_truth"
    / "V5_COMBINED_PREPAIR_FEASIBILITY_GATE_PROTOCOL.json"
)
assert PROTOCOL_PATH.exists(), f"Missing protocol: {PROTOCOL_PATH}"

with open(PROTOCOL_PATH, encoding="utf-8") as f:
    PROTOCOL = json.load(f)

print("ROOT:", ROOT)
print("OUT:", OUT)
print("Protocol:", PROTOCOL_PATH)


ROOT: /Users/sangjinlee/Desktop/projects/saju
OUT: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_combined_prepair_feasibility
Protocol: /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v5_ground_truth/V5_COMBINED_PREPAIR_FEASIBILITY_GATE_PROTOCOL.json


In [2]:

# Locate one exact file by basename.
# For E1 coverage, we will select by the SHA frozen in its manifest.

def find_all_by_name(name):
    matches = sorted(ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"Could not find {name} anywhere under {ROOT}")
    return matches

def load_json_one(name, required_status=None):
    matches = find_all_by_name(name)
    valid = []
    for p in matches:
        try:
            obj = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        if required_status is None or obj.get("status") == required_status:
            valid.append((p, obj))
    if len(valid) != 1:
        raise RuntimeError(
            f"Expected exactly one valid {name}; found {len(valid)}. "
            + "\n".join(str(x[0]) for x in valid)
        )
    return valid[0]

# Original DEV pre-pair failure was frozen before E1 was selected.
orig_path, ORIG = load_json_one(
    "V5_PREPAIR_FEASIBILITY_DECISION.json",
    "V5_PREPAIR_FEASIBILITY_FAIL_EXPANSION_REQUIRED",
)

# E1 final freeze.
e1_dec_path, E1_DEC = load_json_one(
    "V5_E1_EVENT_FREEZE_DECISION.json",
    "V5_E1_EVENT_CORPUS_FROZEN_READY_FOR_COMBINED_PREPAIR_GATE",
)
e1_manifest_path, E1_MAN = load_json_one(
    "V5_E1_EVENT_FREEZE_MANIFEST.json",
    "V5_E1_EVENT_CORPUS_FROZEN_READY_FOR_COMBINED_PREPAIR_GATE",
)

assert sha256_file(e1_manifest_path) == E1_DEC["manifest_sha256"]
assert E1_DEC["combined_prepair_gate_allowed"] is True
assert E1_DEC["pair_generation_allowed"] is False
assert E1_DEC["astrology_generation_allowed"] is False

print("Original prepair decision:", orig_path)
print("E1 decision:", e1_dec_path)
print("E1 manifest:", e1_manifest_path)


Original prepair decision: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_dev_expansion_e1/V5_PREPAIR_FEASIBILITY_DECISION.json
E1 decision: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_dev_expansion_e1_event_freeze/V5_E1_EVENT_FREEZE_DECISION.json
E1 manifest: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_dev_expansion_e1_event_freeze/V5_E1_EVENT_FREEZE_MANIFEST.json


In [3]:

# Resolve the exact E1 subject coverage frozen by the manifest hash.
coverage_matches = find_all_by_name("V5_E1_EVENT_FREEZE_SUBJECT_COVERAGE.csv")
expected_cov_sha = E1_MAN["subject_coverage_sha256"]

matching_cov = [p for p in coverage_matches if sha256_file(p) == expected_cov_sha]
if len(matching_cov) != 1:
    raise RuntimeError(
        f"Expected exactly one E1 subject coverage with SHA {expected_cov_sha}; "
        f"found {len(matching_cov)}"
    )

E1_COV_PATH = matching_cov[0]
e1 = pd.read_csv(E1_COV_PATH)

required_cols = {
    "subject_id", "preassigned_axis",
    "positive_events", "negative_events", "distinct_event_years"
}
missing = required_cols - set(e1.columns)
assert not missing, f"Missing E1 coverage columns: {missing}"

assert len(e1) == int(E1_MAN["E1_subjects_n"]) == 160
assert e1["subject_id"].nunique() == 160

e1["both_polarities"] = (
    (e1["positive_events"] > 0)
    & (e1["negative_events"] > 0)
)

e1_by_axis = (
    e1.groupby("preassigned_axis")["both_polarities"]
      .sum()
      .astype(int)
      .to_dict()
)
e1_total = int(e1["both_polarities"].sum())

print("E1 coverage:", E1_COV_PATH)
print("E1 both-polarity upper bound:", e1_total, e1_by_axis)


E1 coverage: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_dev_expansion_e1_event_freeze/V5_E1_EVENT_FREEZE_SUBJECT_COVERAGE.csv
E1 both-polarity upper bound: 34 {'COMPETITIVE': 19, 'PROJECT': 3, 'STATUS': 12}


In [4]:

# Frozen Original DEV upper bound from the pre-E1 decision.
orig_by_axis = {
    k: int(v)
    for k, v in ORIG["both_polarity_upper_bound_by_axis"].items()
}
orig_total = int(ORIG["both_polarity_upper_bound_total"])

frozen_total = int(PROTOCOL["frozen_min_pairable_total"])
frozen_by_axis = {
    k: int(v)
    for k, v in PROTOCOL["frozen_min_pairable_by_axis"].items()
}

# The protocol must match the thresholds frozen in the pre-E1 decision.
assert orig_total == 71
assert orig_by_axis == {"COMPETITIVE": 31, "PROJECT": 12, "STATUS": 28}
assert int(ORIG["frozen_min_pairable_total"]) == frozen_total
assert {
    k: int(v) for k, v in ORIG["frozen_min_pairable_by_axis"].items()
} == frozen_by_axis

axes = ["COMPETITIVE", "PROJECT", "STATUS"]

combined_by_axis = {
    a: orig_by_axis.get(a, 0) + int(e1_by_axis.get(a, 0))
    for a in axes
}
combined_total = sum(combined_by_axis.values())

summary_rows = []
for a in axes:
    summary_rows.append({
        "axis": a,
        "original_DEV_both_polarity_upper_bound": orig_by_axis[a],
        "E1_both_polarity_upper_bound": int(e1_by_axis.get(a, 0)),
        "combined_both_polarity_upper_bound": combined_by_axis[a],
        "frozen_minimum": frozen_by_axis[a],
        "necessary_condition_pass": combined_by_axis[a] >= frozen_by_axis[a],
        "deficit": max(0, frozen_by_axis[a] - combined_by_axis[a]),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

print("Original upper bound:", orig_total)
print("E1 upper bound:", e1_total)
print("Combined upper bound:", combined_total)
print("Frozen total minimum:", frozen_total)


,axis,original_DEV_both_polarity_upper_bound,E1_both_polarity_upper_bound,combined_both_polarity_upper_bound,frozen_minimum,necessary_condition_pass,deficit
0,COMPETITIVE,31,19,50,20,True,0
1,PROJECT,12,3,15,20,False,5
2,STATUS,28,12,40,25,True,0


Original upper bound: 71
E1 upper bound: 34
Combined upper bound: 105
Frozen total minimum: 80


In [5]:

# Hard safety assertions: this is still PRE-PAIR.
assert ORIG["pair_gap_inspected"] is False
assert ORIG["pairs_generated"] is False
assert ORIG["astrology_generated"] is False
assert ORIG["control_scored"] is False
assert ORIG["confirm_researched"] is False

rules = E1_MAN["rules"]
assert rules["pair_gap_inspected"] is False
assert rules["pairs_generated"] is False
assert rules["chronology_balanced"] is False
assert rules["astrology_generated"] is False
assert rules["control_scored"] is False
assert rules["confirm_researched"] is False

axis_pass = bool(summary["necessary_condition_pass"].all())
total_pass = combined_total >= frozen_total
gate_pass = bool(axis_pass and total_pass)

# Under the frozen evidence currently expected, PROJECT must still fail.
# If this assertion fails, stop and inspect lineage before proceeding.
expected_combined = {"COMPETITIVE": 50, "PROJECT": 15, "STATUS": 40}
assert combined_by_axis == expected_combined, (
    f"Unexpected combined upper bound: {combined_by_axis}; "
    f"expected {expected_combined}. Do not silently continue."
)
assert combined_total == 105
assert gate_pass is False
assert combined_by_axis["PROJECT"] < frozen_by_axis["PROJECT"]

print("Necessary-condition gate PASS?", gate_pass)
print("Remaining deficit:", {
    a: max(0, frozen_by_axis[a] - combined_by_axis[a])
    for a in axes
})


Necessary-condition gate PASS? False
Remaining deficit: {'COMPETITIVE': 0, 'PROJECT': 5, 'STATUS': 0}


In [6]:

# Persist immutable combined feasibility result.
summary_path = OUT / "V5_COMBINED_PREPAIR_FEASIBILITY_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

if gate_pass:
    status = (
        "V5_COMBINED_PREPAIR_FEASIBILITY_PASS_"
        "READY_FOR_LOCAL_PAIRING_NUISANCE_GATE"
    )
    next_rule = PROTOCOL["pass_action"]["action"]
else:
    status = (
        "V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_"
        "EXPANSION_E2_REQUIRED"
    )
    next_rule = PROTOCOL["failure_action"]["action"]

decision = {
    "version": "V5_COMBINED_PREPAIR_FEASIBILITY_DECISION_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "status": status,
    "original_DEV_both_polarity_upper_bound_total": orig_total,
    "original_DEV_both_polarity_upper_bound_by_axis": orig_by_axis,
    "E1_both_polarity_upper_bound_total": e1_total,
    "E1_both_polarity_upper_bound_by_axis": {
        a: int(e1_by_axis.get(a, 0)) for a in axes
    },
    "combined_both_polarity_upper_bound_total": combined_total,
    "combined_both_polarity_upper_bound_by_axis": combined_by_axis,
    "frozen_min_pairable_total": frozen_total,
    "frozen_min_pairable_by_axis": frozen_by_axis,
    "remaining_deficit_by_axis": {
        a: max(0, frozen_by_axis[a] - combined_by_axis[a])
        for a in axes
    },
    "pair_gap_inspected": False,
    "pairs_generated": False,
    "chronology_balanced": False,
    "astrology_generated": False,
    "control_scored": False,
    "confirm_researched": False,
    "thresholds_changed_after_observation": False,
    "project_oversampling_allowed_for_E2": False,
    "E2_whole_wave_required_if_fail": not gate_pass,
    "E2_axis_quotas_if_fail": PROTOCOL["failure_action"]["E2_axis_quotas"],
    "summary_sha256": sha256_file(summary_path),
    "protocol_sha256": sha256_file(PROTOCOL_PATH),
    "original_prepair_decision_sha256": sha256_file(orig_path),
    "E1_freeze_decision_sha256": sha256_file(e1_dec_path),
    "E1_freeze_manifest_sha256": sha256_file(e1_manifest_path),
    "E1_subject_coverage_sha256": sha256_file(E1_COV_PATH),
    "next_rule": next_rule,
}

decision_path = OUT / "V5_COMBINED_PREPAIR_FEASIBILITY_DECISION.json"
decision_path.write_text(
    json.dumps(decision, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(decision, ensure_ascii=False, indent=2))
print()
print("Saved:", summary_path)
print("Saved:", decision_path)


{
  "version": "V5_COMBINED_PREPAIR_FEASIBILITY_DECISION_V1",
  "notebook_version": "SAJU_ML_V5_COMBINED_PREPAIR_FEASIBILITY_GATE_20260817",
  "status": "V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_EXPANSION_E2_REQUIRED",
  "original_DEV_both_polarity_upper_bound_total": 71,
  "original_DEV_both_polarity_upper_bound_by_axis": {
    "COMPETITIVE": 31,
    "PROJECT": 12,
    "STATUS": 28
  },
  "E1_both_polarity_upper_bound_total": 34,
  "E1_both_polarity_upper_bound_by_axis": {
    "COMPETITIVE": 19,
    "PROJECT": 3,
    "STATUS": 12
  },
  "combined_both_polarity_upper_bound_total": 105,
  "combined_both_polarity_upper_bound_by_axis": {
    "COMPETITIVE": 50,
    "PROJECT": 15,
    "STATUS": 40
  },
  "frozen_min_pairable_total": 80,
  "frozen_min_pairable_by_axis": {
    "COMPETITIVE": 20,
    "PROJECT": 20,
    "STATUS": 25
  },
  "remaining_deficit_by_axis": {
    "COMPETITIVE": 0,
    "PROJECT": 5,
    "STATUS": 0
  },
  "pair_gap_inspected": false,
  "pairs_generated": false,
  "chronol


## Expected result for the currently frozen corpora

The frozen evidence should produce:

- Original DEV upper bound: `C31 / P12 / S28 = 71`
- E1 upper bound: `C19 / P3 / S12 = 34`
- Combined upper bound: `C50 / P15 / S40 = 105`
- Frozen minimums: `C20 / P20 / S25`, total `>=80`

Therefore the necessary-condition gate must **FAIL only because PROJECT = 15 < 20**.

Expected status:

```text
V5_COMBINED_PREPAIR_FEASIBILITY_FAIL_EXPANSION_E2_REQUIRED
```

**Do not generate any 1–5y pairs yet.**  
The next action is a whole new outcome-blind `E2` expansion wave using the same `C40 / P50 / S70` quotas. This deliberately avoids changing the axis sampling design after observing that PROJECT remains deficient.
